# Fine-Tuning IndoBERT untuk Deteksi Judi Online
Notebook ini melatih model IndoBERT menggunakan dataset `komentar_bersih.csv`.

In [ ]:
!pip install transformers datasets torch scikit-learn pandas

In [ ]:
import pandas as pd
import torch
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
from datasets import Dataset

In [ ]:
# Muat data
df = pd.read_csv('../komentar_bersih.csv')
df = df.dropna(subset=['teks_bersih', 'label'])
df['label'] = df['label'].astype(int)

# Bagi data
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

In [ ]:
# Load model & tokenizer IndoBERT
model_name = "indolem/indobert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertForSequenceClassification.from_pretrained(model_name, num_labels=2)

In [ ]:
# Tokenisasi
def tokenize_function(examples):
    return tokenizer(examples["teks_bersih"], padding="max_length", truncation=True, max_length=128)

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

In [ ]:
# Training Arguments
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
)

In [ ]:
# Mulai Training (Gunakan GPU/Colab agar cepat)
trainer.train()

In [ ]:
import os
os.makedirs("../model/indobert_judol", exist_ok=True)

# Simpan Model
model.save_pretrained("../model/indobert_judol")
tokenizer.save_pretrained("../model/indobert_judol")
print("Model berhasil disimpan!")